In [1]:
import base64
import requests
import os
import glob
import json
import time
from io import BytesIO
import pandas as pd
from datetime import datetime
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image
from openpyxl import Workbook
from openpyxl.drawing.image import Image as ExcelImage
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
import os.path
from googleapiclient.discovery import build
from google.oauth2 import service_account
import sys
# Define the path to the utils folder
utils_folder = '../utils/'

# Add the utils folder to the system path
sys.path.append(os.path.abspath(utils_folder))
import google_drive_utils as gdu 
import temp_folder_utils as tfu

In [2]:
%env PYTHONPATH=D:/Cardiff_RA_docs/NERD/nerd-data-analysis/

env: PYTHONPATH=D:/Cardiff_RA_docs/NERD/nerd-data-analysis/


In [3]:
# Load environment variables
load_dotenv()

True

In [4]:
# Initialize OpenAI client with API key
client = OpenAI()
OpenAI.api_key = os.getenv('OPENAI_API_KEY')

In [5]:
drive_service  = gdu.create_drive_service()

In [6]:
# Load prompts from the JSON file
with open('../config/prompts.json', 'r') as file:
    prompts = json.load(file)

In [7]:
class Advertisement(BaseModel):
    Image_Link : str
    Ad_Name: str
    Product: str
    Company: str
    Ad_Description: str
    Ad_Placement: str

class AdvertisementList(BaseModel):
    Advertisements: list['Advertisement']

class NewsArticleData(BaseModel):
    Headline: str
    Article_Text: str
    Published_Date: str
    Publishing_Agency_Name: str
    Author_Names: list[str]
    Keywords_Tags: list[str]
    Related_Article_Links: list[str]
    Sentiment: str
    Tone: str
    Narrative: str        

In [8]:
def download_images(image_files):
    """
    Downloads a list of image files from Google Drive.

    Args:
        image_files (list): A list of dictionaries containing image file details. 
                            Each dictionary should have 'File Name' and 'File Link' keys.
    """
    for image in image_files:
        # Extract the file name and link from the dictionary
        file_name = image['file_name']
        file_link = image['file_link']

        # Extract the file ID from the webViewLink
        file_id = gdu.extract_file_id(file_link)

        # Download the file using the extracted file ID
        gdu.download_file_from_drive(file_id, file_name, drive_service)

In [9]:
def encode_images_to_base64(folder_path):
    """
    Load all images from a specified folder and encode them in Base64.

    Args:
        folder_path (str): The path to the folder containing images.

    Returns:
        list: A list of Base64 encoded images.
    """
    images = []
    for filename in glob.glob(os.path.join(folder_path, '*.*')):
        with open(filename, 'rb') as image_file:
            b64_image = base64.b64encode(image_file.read()).decode('utf-8')
            images.append(b64_image)
    return images

In [10]:
def generate_ad_completions(prompt, b64_images,files_data, model='gpt-4o', max_tokens=300):
    """
    Generates advertisement completions based on the provided prompt and base64-encoded images.

    Args:
    prompt (str): The text prompt to generate advertisements.
    b64_images (list): A list of base64-encoded images.
    model (str): The model to use for generating completions. Default is 'gpt-4o'.
    max_tokens (int): The maximum number of tokens for the response. Default is 300.

    Returns:
    list: A list of advertisement completions.
    """
    ad_completions = []  # Initialize an empty list to store the advertisement completions
    
    # Loop through each base64-encoded image
    for b64_image , file in zip(b64_images,files_data):
        # Generate a chat completion using the provided model
        completion = client.beta.chat.completions.parse(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}"
                            }
                        }
                    ]
                }
            ],
            response_format=AdvertisementList,
            max_tokens=max_tokens
        )
        
        # Parse the completion result and append to the list
        ad_content = completion.choices[0].message.parsed
        for ad in ad_content.Advertisements:
            ad.Image_Link = file['file_link']
        ad_completions.append(ad_content)
    
    return ad_completions  # Return the list of advertisement completions


In [11]:
def generate_news_article_completions(prompts, b64_images, model='gpt-4o', max_tokens=2000):
    """
    Sends a request to the OpenAI API to generate news article completions based on the given prompts
    and base64-encoded images.

    Args:
        prompts (list): A list of text prompts to be used by the model.
        b64_images (list): A list of base64-encoded images.
        model (str): The model name to be used. Defaults to 'gpt-4o'.
        max_tokens (int): The maximum number of tokens to generate. Defaults to 2000.

    Returns:
        dict: A parsed response containing the generated news article.
    """ 
    # Create the payload for the API request
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt} for prompt in prompts
                ] + [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{b64_image}"
                        }
                    } for b64_image in b64_images
                ]
            }
        ],
        response_format=NewsArticleData,
        max_tokens=max_tokens
    )

    # Return the parsed message containing the generated news article
    return completion.choices[0].message.parsed

In [12]:
def generate_dataframe_from_folders(parent_folder_id, prompts,temp_folder_path = "../screenshot_data/temp/"):
    """
    Generate DataFrames containing article and advertisement data by processing folders with images and prompts.

    Args:
        root_path (str): The root directory containing folders with images.
        prompts (dict): A dictionary containing text prompts for news article and advertisement extraction.

    Returns:
        tuple: Two DataFrames, one containing the processed article data and the other containing advertisement data.
    """
    articles_data = []
    ads_data = []
    folders_list  = gdu.list_folders_in_folder(drive_service,parent_folder_id)
    tfu.delete_temp_folder()
    # Iterate through each folder in the root directory
    for folder in folders_list:
        files_data =  gdu.list_files_in_folder(drive_service,folder['folder_id'])
        tfu.create_temp_folder()
        download_images(files_data)
        # Load all images from the folder and encode them to base64
        b64_images = encode_images_to_base64(temp_folder_path)
        # Generate news article data using the corresponding prompts
        news_article_data = generate_news_article_completions(
            prompts["news_article_data_extraction_prompts"], b64_images,  model='gpt-4o-mini', max_tokens=2000
        )
        
        # Generate advertisement data using the corresponding prompt
        advertisement_data = generate_ad_completions(
            prompts["advertisement_extraction_prompt"], b64_images, files_data, model='gpt-4o-mini', max_tokens=2000
        )
        
        # Process and store the advertisement data
        for ads in advertisement_data:
            for ad in ads.Advertisements:
                ad_dict = {
                    'Folder_Number': folder['folder_name'],
                    'Image_Link': ad.Image_Link,
                    'Ad_Name': ad.Ad_Name,
                    'Product': ad.Product,
                    'Company': ad.Company,
                    'Ad_Description': ad.Ad_Description,
                    'Ad_Placement': ad.Ad_Placement
                }
                ads_data.append(ad_dict)
        
        # Construct and store the article data for the current folder
        article_dict = {
            'Folder_Number': folder['folder_name'],
            'Image_Links':[file['file_link'] for file in files_data],
            'Headline': news_article_data.Headline,
            'Article_Text': news_article_data.Article_Text,
            'Published_Date': news_article_data.Published_Date,
            'Publishing_Agency_Name': news_article_data.Publishing_Agency_Name,
            'Author_Names': news_article_data.Author_Names,
            'Keywords_Tags': news_article_data.Keywords_Tags,
            'Related_Article_Links': news_article_data.Related_Article_Links,
            'Sentiment': news_article_data.Sentiment,
            'Tone': news_article_data.Tone,
            'Narrative': news_article_data.Narrative
        }
        articles_data.append(article_dict)
        tfu.delete_temp_folder()
    # Return DataFrames containing the processed article and advertisement data
    return pd.DataFrame(articles_data), pd.DataFrame(ads_data)


In [13]:
grand_parent_id  = "11McLwHRxYiHpOCj3v0emz47MpuJMP3Kr"
main_folders_list  = gdu.list_folders_in_folder(drive_service,grand_parent_id)

In [17]:
for main_folder in  main_folders_list:
    article_data_df, ad_data_df = generate_dataframe_from_folders(main_folder['folder_id'], prompts,temp_folder_path = "../screenshot_data/temp/")
    date_object = datetime.strptime(main_folder['folder_name'], "%d/%m/%Y")
    formatted_date = date_object.strftime("%d-%m-%Y")
    article_data_df.to_csv(f"../datasets/articles_dataset_{formatted_date}.csv",index=False)
    ad_data_df.to_csv(f"../datasets/ads_dataset_{formatted_date}.csv",index=False)
    print("done")

done
done
done


In [16]:
main_folders_list.remove({'folder_name': '07/06/2024',
  'folder_id': '1xmAESFDfBV6JM7w1CwbYC3KebMEU2e9f'})